::: {.sb-lesson-links}
[Exercises →](../exercises/california-housing.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/02-linear-regression.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-performance-and-feature-selection.ipynb){download="lesson-performance-and-feature-selection.ipynb"}  
Data: [california_housing_market.csv](../downloads/data/california_housing_market.csv){download="california_housing_market.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

The main feature of multiple linear regression is that it uses several explanatory variables together. Therefore, the natural question is: which variables should I use to build the best possible model for my objective? This question leads us to introduce model evaluation criteria and variable selection methods.

## What will you learn in this course? 🧐🧐

This course focuses on evaluation methods for multiple linear regression models.

* Evaluation of multiple linear regression models
    * Analysis of Variance (ANOVA)
    * Fisher's F-statistic
    * $R^{2}$ (R squared)
    * $R^2_{adjusted}$
    * P-values
* Model selection
    * Step-by-step methods
* Final remarks

The demos 🔬 of this lesson use the California housing dataset (target: `MedHouseVal`, the median house value of each district). The next cell loads the libraries and the data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

housing = pd.read_csv("../data/california_housing_market.csv")
features = housing.drop(columns="MedHouseVal")
y = housing["MedHouseVal"]

## Evaluation of multiple linear regression models 💯

Some of the evaluation criteria presented below can also be used for models other than multiple linear regression. This is why it is important to introduce them now and to remember how to interpret each of them.


### Analysis of Variance (ANOVA)

The analysis of variance quantifies the performance of a statistical model in terms of estimation error. The values we discuss now will be used to build other performance metrics:



::: {.callout-tip title="Intuition"}
Think of the variation of the target as a budget. House values differ from one district to another: that's the total budget, SST. A model "explains" a part of these differences (SSE, for example "this district is expensive because incomes are high"), and the rest is left unexplained (SSR, the errors). The three quantities are linked by a simple relation: $SST = SSE + SSR$ (for a linear regression with an intercept, on its training data).
:::



* SST: the Sum of Squares Total measures the dispersion of the values of the target variable $Y$ (denoted $y_{1}, ..., y_{n}$) over the population considered. Mathematically:

$$
SST = \sum_{i=1}^{n}(y_{i}-\bar{y})^2
$$

It is the sum of the squared deviations from the mean of the target variable $Y$ for the $n$ observations considered.

* SSE: the Sum of Squares Explained represents the amount of dispersion of the target variable that is explained by the model. It is defined as:

$$
SSE = \sum_{i=1}^{n}(\hat{y}_{i}-\bar{y})^2
$$

It is the sum of the squared differences between the model's estimate for each observation and the mean of the target variable over the population of interest. In statistics, variation is information: you cannot possibly differentiate samples if they are all described by the exact same set of values.

* SSR: the Sum of Squared Residuals quantifies the error made by the model, in other words the part of the dispersion of the target variable that is not explained by the model (hence the name "residual"). Its formula is:

$$
SSR =\sum_{i=1}^{n}(y_{i}-\hat{y}_{i})^2
$$

It is essential to understand these values, because we will use them to build all the evaluation metrics for multiple linear regression that we will see now.

To summarize, SST (in red in the figure below) is proportional to the total variance of the target variable, which can be decomposed into two components:

* SSE is the variance explained by the model, proportional to the variance of our estimates around the actual mean of the observed population;
* SSR (in blue in the figure below) is the sum of the squared differences between our estimates and the actual values of the target variable.

In other words, SST is the total amount of information, SSE is the information explained by the model, and SSR is the information that remains to be explained, i.e. the error made.


![SSR_SST](../img/ssr_sst.png)

#### See it with data 🔬: the variance budget of a real model

We fit a linear regression with the 8 variables of the housing dataset, and compute SST, SSE and SSR.

In [ ]:
model = LinearRegression().fit(features, y)
y_hat = model.predict(features)

SST = np.sum((y - y.mean()) ** 2)
SSE = np.sum((y_hat - y.mean()) ** 2)
SSR = np.sum((y - y_hat) ** 2)
print(f"SST = {SST:,.0f}")
print(f"SSE = {SSE:,.0f}  ({SSE / SST:.1%} of SST)")
print(f"SSR = {SSR:,.0f}  ({SSR / SST:.1%} of SST)")
print(f"SSE + SSR = {SSE + SSR:,.0f}")

SST = 27,483
SSE = 16,661  (60.6% of SST)
SSR = 10,822  (39.4% of SST)
SSE + SSR = 27,483


The model explains 60.6% of the variation of house values (SSE = 16,661) and leaves 39.4% unexplained (SSR = 10,822). The two parts add up exactly to the total, SST = 27,483. Keep the 60.6% in mind: you're about to meet it again under another name.

* $R^{2}$ (R squared)

$R^{2}$, or R-squared, is a statistic that quantifies the explanatory power of the model with respect to the target variable.

$$
R^{2} = 1-\frac{SSR}{SST}
$$

$R^2$ varies between 0 and 1. If the model is not very relevant, the sum of squared residuals $SSR$ is close to the total sum of squares $SST$, and $R^{2}$ is close to 0. On the contrary, if the model explains the target variable faithfully, then $SSR$ is close to 0 and $R^2$ is close to 1.

$R^2$ never decreases when you add an explanatory variable: mechanically, each new variable makes the prediction of the target variable $Y$ on the training data at least as good, so $R^2$ gets higher (or stays the same). This is why $R^2$ should only be used to compare models that have the same number of explanatory variables.

#### See it with data 🔬: $R^2$ by hand, and why it can fool you

First, let's check that $1 - SSR/SST$ is exactly what scikit-learn's `.score()` returns:

In [ ]:
print(f"R² by hand     : {1 - SSR / SST:.4f}")
print(f"R² from sklearn: {model.score(features, y):.4f}")

R² by hand     : 0.6062
R² from sklearn: 0.6062


Now let's verify the claim "$R^2$ never decreases when you add a variable", with variables that are **pure noise** (random numbers, unrelated to house values). We train on a small sample of 100 districts with `MedInc` only, then add noise columns one at a time, up to 40. For each model, we compute:

* the $R^2$ on the training data;
* the adjusted $R^2$, which penalizes the number of variables $p$ (its formula, $1-\frac{n-1}{n-p-1}(1-R^2)$, is explained in the optional lesson on computing metrics);
* the $R^2$ on 1,000 other districts (the test set).

In [ ]:
rng = np.random.default_rng(0)
train = housing.sample(100, random_state=0)
test = housing.drop(train.index).sample(1000, random_state=0)
noise_train, noise_test = rng.normal(size=(100, 40)), rng.normal(size=(1000, 40))

rows = []
for k in range(41):
    X_tr = np.column_stack([train[["MedInc"]], noise_train[:, :k]])
    X_te = np.column_stack([test[["MedInc"]], noise_test[:, :k]])
    m = LinearRegression().fit(X_tr, train["MedHouseVal"])
    r2_train, p = m.score(X_tr, train["MedHouseVal"]), X_tr.shape[1]
    rows.append({"noise variables": k, "R² (train)": r2_train,
                 "adjusted R² (train)": 1 - (100 - 1) / (100 - p - 1) * (1 - r2_train),
                 "R² (test)": m.score(X_te, test["MedHouseVal"])})
scores = pd.DataFrame(rows)

In [ ]:
fig = px.line(scores, x="noise variables", y=["R² (train)", "adjusted R² (train)", "R² (test)"],
              title="Adding pure-noise variables: R² goes up, the real quality goes down",
              labels={"noise variables": "Number of pure-noise variables added", "value": "Score", "variable": ""})
fig.show()
adj = scores["adjusted R² (train)"]
print(f"Adjusted R² stays between {adj.min():.3f} and {adj.max():.3f}")
scores.iloc[[0, 10, 20, 40]].round(3)

Adjusted R² stays between 0.379 and 0.460


,noise variables,R² (train),adjusted R² (train),R² (test)
0,0,0.404,0.398,0.435
10,10,0.467,0.400,0.363
20,20,0.568,0.451,0.180
40,40,0.642,0.389,-0.173


Both computations give 0.6062: the $R^2$ is exactly the 60.6% share of SST explained by the model that we computed above.

The experiment with noise variables confirms the warning of the lesson:

* The training $R^2$ climbs from 0.404 (income only) to 0.642 with 40 noise variables, although these variables contain no information at all: the model uses them to fit the random fluctuations of its 100 training districts.
* The test $R^2$ tells the real story: it falls from 0.435 to 0.180 with 20 noise variables, and to -0.173 with 40 (worse than always predicting the average!).
* The adjusted $R^2$ cancels the illusory gain: it stays between 0.379 and 0.460 instead of climbing. It doesn't detect the damage measured on the test set, though, so it's a useful correction, not a replacement for a test set.

## Model selection 🤔🤔

When we have $p$ explanatory variables, we can count the number of possible models in the following way: each explanatory variable can be either in the model or not, so we can potentially build $2^p$ models. In practice, when the number $p$ of explanatory variables is large, we cannot reasonably explore all the $2^p$ models to select the best one. Several methods exist to avoid this brute-force approach.



::: {.callout-tip title="Intuition"}
With 8 variables there are already $2^8 = 256$ possible models; with 30 variables, more than a billion. Step-by-step methods work like a greedy shopper: at each step they make the choice that looks best right now (add the most useful variable, or remove the most useless one), without exploring every combination. It's fast, and usually good enough, but it is not guaranteed to find the very best model.
:::



### Step by step

Step-by-step selection comes in three variants:

* Forward selection: the variables are added to the model one by one. At each step, we add the variable that maximizes the score on the test set ($R^2$ in the case of linear regression). The procedure stops when all the variables are used, or when the score on the test set starts decreasing.
* Backward elimination: this time, we start with a model using all the explanatory variables. At each step, we remove the variable with the highest p-value (the p-value of Student's t-test on its coefficient). The procedure stops when all the remaining variables have p-values lower than a threshold, set by default at 0.05 (it can be adapted to the precision needs of the problem). This technique is difficult to implement with scikit-learn, since the p-values have to be calculated by hand.
* Stepwise: this algorithm alternates between selection and elimination. After each addition of a variable, an elimination step removes the variables that have become less relevant in the presence of the new one. For example, if after adding the most useful variable (according to the test score) one of the variables becomes non-significant, we remove it and proceed with a new forward selection step.

#### See it with data 🔬: forward selection on the housing data

We split the data into a training set and a test set, then apply forward selection exactly as described above: at each step, we try every remaining variable, and keep the one that gives the best $R^2$ on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.2, random_state=0)

selected, remaining, path = [], list(features.columns), []
while remaining:
    tries = {v: LinearRegression().fit(X_train[selected + [v]], y_train).score(X_test[selected + [v]], y_test)
             for v in remaining}
    best = max(tries, key=tries.get)
    selected.append(best)
    remaining.remove(best)
    path.append({"step": len(selected), "variable added": best, "test R²": round(tries[best], 4)})
path = pd.DataFrame(path)
path

,step,variable added,test R²
0,1,MedInc,0.4467
1,2,HouseAge,0.4838
2,3,Latitude,0.4933
3,4,Longitude,0.5765
4,5,AveBedrms,0.5825
5,6,AveRooms,0.5919
6,7,AveOccup,0.5945
7,8,Population,0.5943


In [ ]:
fig = px.line(path, x="step", y="test R²", text="variable added", markers=True,
              title="Forward selection: test R² after each added variable",
              labels={"step": "Number of variables in the model", "test R²": "R² on the test set"})
fig.update_traces(textposition="bottom right")
fig.show()

The procedure first picks `MedInc` (test $R^2$ = 0.4467), the most informative variable on its own. Notice the big jump at step 4: `Latitude` alone brings little (0.4838 → 0.4933), but together with `Longitude` it pins down the location of the district (0.4933 → 0.5765). At the last step, adding `Population` makes the test score slightly decrease (0.5945 → 0.5943): following the rule above, we would stop at 7 variables and leave `Population` out.

#### See it with data 🔬: backward elimination with p-values

For backward elimination we need the p-values of the coefficients, which `statsmodels` computes for us (the optional lesson on computing metrics explains how). To make things interesting, we use a sample of 300 districts and add two pure-noise variables, `noise_1` and `noise_2`. At each step we remove the variable with the highest p-value, until all p-values are below 0.05.

In [ ]:
sample = housing.sample(300, random_state=1)
X_back = sample.drop(columns="MedHouseVal").assign(
    noise_1=np.random.default_rng(1).normal(size=300), noise_2=np.random.default_rng(2).normal(size=300))

while True:
    ols = sm.OLS(sample["MedHouseVal"], sm.add_constant(X_back)).fit()
    pvalues = ols.pvalues.drop("const")
    if pvalues.max() < 0.05:
        break
    print(f"remove {pvalues.idxmax():11s} (p-value = {pvalues.max():.3f})")
    X_back = X_back.drop(columns=pvalues.idxmax())
print("\nKept:", list(X_back.columns))
ols.pvalues.drop("const").round(4)

remove AveBedrms   (p-value = 0.974)
remove noise_2     (p-value = 0.905)
remove AveRooms    (p-value = 0.545)
remove noise_1     (p-value = 0.408)
remove Population  (p-value = 0.148)
remove HouseAge    (p-value = 0.241)

Kept: ['MedInc', 'AveOccup', 'Latitude', 'Longitude']


MedInc       0.0
AveOccup     0.0
Latitude     0.0
Longitude    0.0
dtype: float64

The two noise variables are eliminated (p-values 0.905 and 0.408), together with 4 real variables that don't bring significant information **once the others are in the model** on this small sample (`AveBedrms`, `AveRooms`, `Population`, `HouseAge`). Notice that p-values change at each step: the p-value of `HouseAge` was below 0.148 while `Population` was in the model, but it rose to 0.241 once `Population` was removed. This is why variables are removed one at a time. The final model keeps 4 variables, all with p-values below 0.0001 (displayed as 0.0).

### Final Notes

The model evaluation and selection methods introduced above are valid for all linear models, as well as for logistic regression, which we will discuss later.

## Resources 📚📚

* How do I interpret R Squared - https://blog.minitab.com/en/blog/adventures-in-statistics-2/regression-analysis-how-do-i-interpret-r-squared-and-assess-the-goodness-of-fit
* Adjusted R Squared - https://blog.minitab.com/en/blog/adventures-in-statistics-2/multiple-regession-analysis-use-adjusted-r-squared-and-predicted-r-squared-to-include-the-correct-number-of-variables

::: {.sb-next}
**[Next: Optional: maximum likelihood →](../lessons/maximum-likelihood.ipynb)**
:::